In [8]:
%%writefile practice10_1.cpp
// Practice 10 Task 1:  Анализ производительности CPU-параллельной программы (OpenMP)
// реализовать базовую параллельную версию;
// выполнить профилирование программы с использованием omp_get_wtime() и/или профилировщика (Intel VTune, gprof);
// определить:
   // долю параллельной и последовательной части программы;
   // влияние числа потоков на ускорение;
// проанализировать результаты в контексте закона Амдала.

#include <iostream>        // Для ввода-вывода
#include <vector>          // Для использования динамического массива (vector)
#include <cmath>           // Для математических операций (pow)
#include <omp.h>           // Библиотека OpenMP

int main() {               // Основная функция

    const int N = 10'000'000;      // Размер массива (10 миллионов элементов)

    std::vector<double> data(N);   // Создание массива данных


    for (int i = 0; i < N; i++) {  // Заполнение массива значениями
        data[i] = i * 0.001;       // Простые данные для вычислений
    }

    // Переменные для результатов
    double sum = 0.0;             // Сумма
    double mean = 0.0;            // Среднее значение
    double variance = 0.0;        // Дисперсия



    double start = omp_get_wtime(); // Замер начала выполнения
    #pragma omp parallel for reduction(+:sum) // ПАРАЛЛЕЛЬНОЕ ВЫЧИСЛЕНИЕ СУММЫ
    for (int i = 0; i < N; i++) {
        sum += data[i];                       // reduction — автоматическое объединение результатов
    }

    // Вычисление среднего (последовательная часть)
    mean = sum / N;

    // ПАРАЛЛЕЛЬНОЕ ВЫЧИСЛЕНИЕ ДИСПЕРСИИ
    #pragma omp parallel for reduction(+:variance)
    for (int i = 0; i < N; i++) {                       // Цикл по всем элементам массива
        variance += pow(data[i] - mean, 2);             // Вычисляем квадрат отклонения текущего элемента от среднего значения
                                                        // (data[i] - mean) — отклонение
                                                        // pow(..., 2) — возведение отклонения в квадрат
                                                        // Результат добавляется к локальной переменной variance каждого потока
    }


    variance /= N;               // Финальное вычисление дисперсии

    double end = omp_get_wtime(); // Замер конца выполнения


    // Вывод результатов
    std::cout << "Сумма = " << sum << std::endl;
    std::cout << "Среднее значение = " << mean << std::endl;
    std::cout << "Дисперсия = " << variance << std::endl;
    std::cout << "Время выполнения = " << (end - start) << " с" << std::endl;

    return 0;
}


Overwriting practice10_1.cpp


In [13]:
!g++ -fopenmp practice10_1.cpp -o practice10_1


In [14]:
!OMP_NUM_THREADS=1 ./practice10_1


Сумма = 5e+10
Среднее значение = 5000
Дисперсия = 8.33333e+06
Время выполнения = 0.359145 с


In [15]:
!OMP_NUM_THREADS=2 ./practice10_1


Сумма = 5e+10
Среднее значение = 5000
Дисперсия = 8.33333e+06
Время выполнения = 0.303681 с


In [16]:
!OMP_NUM_THREADS=4 ./practice10_1


Сумма = 5e+10
Среднее значение = 5000
Дисперсия = 8.33333e+06
Время выполнения = 0.287411 с
